# 03 · Validation and guardrails

**Role of this layer.** "The LLM proposes, the system disposes." A plan is inert
JSON; the validator decides, *without touching any data values*, whether it may run.
Every issue has an **action**, what happens next:

* **BLOCK**: the run stops (unknown capability, bad parameter, returns of a yield...).
* **APPROVE**: a human must decide; it runs only after explicit approval (look-ahead,
  material assumptions).
* **WARN**: the run continues and the result is flagged (shown, passed to the narrator).

Each guardrail's code also names the pipeline stage that enforces it: **R** the
request check (before the planner), **P** the planner (LLM), **V** the validator
(this notebook), **E** the executor (notebook 04), **N** the narrator check
(notebook 06).

It does a metadata *dry run*: walking the steps and propagating data type, kind
and frequency through the registry, which is how it catches "returns of a yield"
or "the spread of a daily and a weekly series" before anything executes.

Below, every validator guardrail is triggered on a deliberately broken plan, and
the last section edits a plan's parameters to show the same validator
re-checking every change (V09 appearing and clearing, a frequency edit, an
out-of-range BLOCK, a rejected structural edit).
The app's **Guardrails** page always shows the current list of guardrails.

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
import copy

from quantlab.data import load_catalog
from quantlab.guardrails import GUARDRAILS, check_schema, fill_defaults, validate_plan
from quantlab.mock_scenarios import SCENARIOS_BY_ID
from quantlab.plan import Plan

catalog = load_catalog(settings.catalog_path)


def scenario_plan(scenario_id):
    """A mutable copy of a demo scenario's canned plan."""
    s = SCENARIOS_BY_ID[scenario_id]
    return {"plan_id": "demo", "request": s.request, **copy.deepcopy(s.plan)}


# A clean rates plan (not one of the example workflows): the 10y-2y spread and its changes.
SPREAD = {
    "dataset_ids": ["DGS10", "DGS2"],
    "steps": [
        {"id": "s1", "capability": "load_series", "params": {"dataset_id": "DGS10"}},
        {"id": "s2", "capability": "load_series", "params": {"dataset_id": "DGS2"}},
        {"id": "s3", "capability": "spread", "inputs": {"a": "s1", "b": "s2"}},
        {
            "id": "s4",
            "capability": "difference",
            "inputs": {"series": "s3"},
            "params": {"periods": 1},
        },
    ],
}


def spread_plan():
    """A mutable copy of the spread plan."""
    return {"plan_id": "demo", "request": "10y-2y spread", **copy.deepcopy(SPREAD)}


def codes_for(raw, **budgets):
    """Parse and validate a plan dict; return the issue codes raised."""
    plan, schema_issues = check_schema(raw)
    if plan is None:
        return sorted({i.code for i in schema_issues})
    return validate_plan(fill_defaults(plan), catalog, **budgets).codes()


pd.DataFrame(
    [
        {
            "code": g.code,
            "name": g.name,
            "action": g.action.value,
            "enforced by": g.enforced_by,
            "type": g.type.value,
            "category": g.category.value,
        }
        for g in GUARDRAILS.values()
    ]
)

,code,name,action,enforced by,type,category
0,R01,REQUEST_TOO_LONG,BLOCK,Request check,Deterministic,Input validity
1,P01,PLANNER_REFUSED,BLOCK,Planner,LLM,Approved scope
2,V01,INVALID_PLAN,BLOCK,Validator,Deterministic,Plan integrity
3,V02,UNKNOWN_CAPABILITY,BLOCK,Validator,Deterministic,Approved scope
4,V03,INVALID_PARAM,BLOCK,Validator,Deterministic,Input validity
5,V04,TYPE_MISMATCH,BLOCK,Validator,Deterministic,Input validity
6,V05,BUDGET_EXCEEDED,BLOCK,Validator,Deterministic,Plan integrity
7,V06,UNKNOWN_DATASET,BLOCK,Validator,Deterministic,Approved scope
8,V07,RETURNS_ON_RATES,BLOCK,Validator,Deterministic,Methodology
9,V08,FREQUENCY_MISMATCH,BLOCK,Validator,Deterministic,Methodology


A clean plan raises nothing; scenario 5 only needs the return-method approvals (of its
returns and of its forward returns).

In [3]:
print("spread plan:", codes_for(spread_plan()))
print("scenario 5:", codes_for(scenario_plan("buy_the_dip_tech")))

spread plan: []
scenario 5: ['V10']


Now one crafted plan per guardrail:

In [4]:
S5 = "buy_the_dip_tech"
cases = {}

raw = scenario_plan(S5)
raw["steps"][0]["code"] = "import os"
cases["V01 extra field 'code'"] = raw

raw = scenario_plan(S5)
raw["steps"][3]["inputs"] = {"series": "s5"}
cases["V01 reference to a later step"] = raw

raw = scenario_plan(S5)
raw["steps"].append({"id": "s9", "capability": "arima", "inputs": {"series": "s3"}})
cases["V02 a planner inventing an ARIMA model"] = raw

raw = scenario_plan(S5)
raw["steps"][3]["params"]["min_periods"] = 5
cases["V03 min_periods below range"] = raw

raw = scenario_plan(S5)
raw["steps"][4]["inputs"] = {"series": "s4"}
cases["V04 forward return of the flags"] = raw

raw = scenario_plan(S5)
raw["steps"][6]["inputs"]["outcome"] = "s2"
cases["V04 significance test on a price level"] = raw

raw = spread_plan()
raw["steps"] += [
    {
        "id": f"s{i}",
        "capability": "difference",
        "inputs": {"series": "s3"},
        "params": {"periods": 1},
    }
    for i in range(5, 17)
]
cases["V05 16 steps > budget of 15"] = raw

raw = spread_plan()
raw["steps"][1]["params"] = {"dataset_id": "BITCOIN"}
cases["V06 dataset not in catalog"] = raw

raw = spread_plan()
raw["steps"] = [
    raw["steps"][0],
    {
        "id": "s2",
        "capability": "compute_returns",
        "inputs": {"series": "s1"},
        "params": {"method": "simple"},
    },
]
cases["V07 returns of a yield"] = raw

cases["V08 weekly 10y minus daily 2y yield (scenario 2)"] = scenario_plan(
    "spread_frequency_mismatch_block"
)
raw = scenario_plan("chf_safe_haven_success")
raw["steps"][4]["params"]["method"] = "full_sample"
cases["V09 full-sample VIX threshold (scenario 1, edited)"] = raw

cases["V10 return methods need approval (scenario 5)"] = scenario_plan(S5)
cases["P01 planner refusal (scenario 3)"] = scenario_plan("arima_forecast_block")
cases["V02 simulated planner follows an injection (scenario 4)"] = scenario_plan(
    "prompt_injection_unknown_capability"
)

raw = scenario_plan(S5)
raw["steps"].append({"id": "s8", "capability": "flag_regime", "inputs": {"series": "s5"}})
cases["V11 flags built from forward returns"] = raw

pd.DataFrame(
    [{"case": name, "codes raised": ", ".join(codes_for(raw))} for name, raw in cases.items()]
)

,case,codes raised
0,V01 extra field 'code',V01
1,V01 reference to a later step,"V01, V10"
2,V02 a planner inventing an ARIMA model,"V02, V10"
3,V03 min_periods below range,"V03, V10"
4,V04 forward return of the flags,"V04, V10"
5,V04 significance test on a price level,"V04, V10"
6,V05 16 steps > budget of 15,V05
7,V06 dataset not in catalog,V06
8,V07 returns of a yield,"V07, V10"
9,V08 weekly 10y minus daily 2y yield (scenario 2),"V08, V09"


The request check comes first, before the planner is even called: a request over
the limits is refused with **R01 REQUEST_TOO_LONG** and never reaches the LLM (the
engine calls this check in `Session.create_plan`).

In [5]:
from quantlab.guardrails import MAX_REQUEST_WORDS, check_request_length

for text in ["Flag the high-volatility weeks of the NASDAQ.", "word " * (MAX_REQUEST_WORDS + 1)]:
    issues = check_request_length(text)
    found = [f"[{i.action}] {i.code}: {i.message}" for i in issues]
    print(f"{len(text.split()):>5} words ->", found or "passes")

    7 words -> passes
 1001 words -> ['[BLOCK] R01: The request has 1,001 words and 5,005 characters; the limits are 1,000 words and 8,000 characters.']


Each issue carries a message and a suggestion. For example, scenario 2 (the spread of a
weekly and a daily yield):

In [6]:
report = validate_plan(
    fill_defaults(Plan(**scenario_plan("spread_frequency_mismatch_block"))), catalog
)
for issue in report.issues:
    print(f"[{issue.action}] {issue.issue_id}: {issue.message}\n    -> {issue.suggestion}")

[BLOCK] V08:s4: spread inputs have different frequencies: 'a' is weekly, 'b' is daily.
    -> Resample DGS2 to weekly right after load_series (resample(freq='W')), before computing the 'b' input.
[APPROVE] V09:s5: A full-sample threshold judges past weeks using information from the future: the threshold (the 10% quantile) is computed from every week, including those that came later, so a week flagged here could not have been flagged at the time (look-ahead bias). Results will look better than anything achievable in real time.
    -> Use method='expanding': each threshold then uses only the data before that week. Keep full_sample only for purely descriptive labelling.


The dry run: predicted output metadata of each step of scenario 5 (no data touched).

In [7]:
report = validate_plan(fill_defaults(Plan(**scenario_plan(S5))), catalog)
columns = ["data_type", "kind", "frequency", "return_method", "horizon", "dataset_ids"]
pd.DataFrame(report.dry_run).T[columns]

,data_type,kind,frequency,return_method,horizon,dataset_ids
s1,series,price_level,D,None,None,[NASDAQCOM]
s2,series,price_level,W,None,None,[NASDAQCOM]
s3,series,returns,W,simple,None,[NASDAQCOM]
s4,flags,flags,W,simple,None,[NASDAQCOM]
s5,series,forward_returns,W,simple,1,[NASDAQCOM]
s6,table,None,W,simple,1,[NASDAQCOM]
s7,table,None,W,simple,1,[NASDAQCOM]


### Editing a plan: every change goes through the same validator

A user may change parameter VALUES only, through the engine's `apply_param_edits`.
Each edit is applied to a copy of the plan and the full validator runs again, so
issues appear and disappear; any edit also resets the approvals.

In [8]:
from quantlab.engine import Session

example = SCENARIOS_BY_ID[S5]
session = Session.start(example.request, list(example.dataset_ids), "mock", settings)
print("as proposed:          ", session.create_plan().codes())
print(
    "method -> full_sample:", session.apply_param_edits({"s4": {"method": "full_sample"}}).codes()
)
print("method -> expanding:  ", session.apply_param_edits({"s4": {"method": "expanding"}}).codes())

as proposed:           ['V10']
method -> full_sample: ['V09', 'V10']
method -> expanding:   ['V10']


Switching the resample step from weekly to daily raises nothing new: the minimum
history (`min_periods`) keeps its value, which now counts trading days instead of
weeks (the registry describes it in periods of the input's frequency, and the app
labels it so). The dry run follows the edit:

In [9]:
report = session.apply_param_edits({"s2": {"freq": "D"}})
print("freq -> D:", report.codes())
pd.DataFrame(report.dry_run).T.loc[["s2", "s4", "s5"], ["kind", "frequency"]]

freq -> D: ['V10']


,kind,frequency
s2,price_level,D
s4,flags,D
s5,forward_returns,D


The widgets in the app are bounded, but the validator is the backstop: an
out-of-range value is a BLOCK, and a structural edit is rejected outright.

In [10]:
for edits in [{"s4": {"quantile": 1.5}}, {"s4": {"capability": "arima"}}]:
    session.reset_to_proposal()  # start each example from the planner's proposal
    report = session.apply_param_edits(edits)
    for issue in report.blocks:
        print(f"{edits} -> [{issue.action}] {issue.code}: {issue.message}")
print("after a reset:", session.reset_to_proposal().codes())

{'s4': {'quantile': 1.5}} -> [BLOCK] V03: 'quantile'=1.5 is outside [0.01, 0.99].


{'s4': {'capability': 'arima'}} -> [BLOCK] V03: Edit rejected: 'capability' is not a parameter of s4 (flag_regime).
after a reset: ['V10']
